In [ ]:
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Paramètres à adapter au jeu de données
# ---------------------------------------------------------------------------
DATA_PATH = "data_path" # <- chemin du jeu de données
TARGET_COL = "target_col" # <- variable cible
ID_COLS = ["id"] # <- identifiants (exclus des analyses et du modèle)
POSITIVE_CLASS = "" # <- classe d'intérêt (classification)

# Analyse Exploratoire des données

In [ ]:
# ---------------------------------------------------------------------------
# Charte graphique Wavestone (reprise du template use_case2_wavestone)
# ---------------------------------------------------------------------------
COLORS = {
    "primary": "#3C18AE",       # violet profond Wavestone
    "secondary": "#441DC6",     # violet vif Wavestone
    "accent": "#8B7FF0",        # violet clair
    "neutral": "#DFD6C5",       # sable / beige
    "neutral_dark": "#C0B5B3",  # gris chaud
    "bg_light": "#EBEEFF",      # bleu-violet très clair
    "dark": "#1A1A2E",          # anthracite (titres / contraste)
    "grey": "#6B6B7B",          # texte secondaire
    "positive": "#2E8B57",      # vert (classe / modalité favorable)
    "negative": "#B23A48",      # rouge (classe / modalité défavorable)
}

SEQUENCE = [
    COLORS["primary"], COLORS["accent"], COLORS["neutral"],
    COLORS["secondary"], COLORS["neutral_dark"], COLORS["grey"],
]

TEMPLATE_LAYOUT = {
    "paper_bgcolor": "white",
    "plot_bgcolor": "white",
    "font": {"family": "Calibri, Arial, sans-serif", "size": 13, "color": COLORS["dark"]},
    "title": {"font": {"size": 20, "color": COLORS["dark"]}},
    "colorway": SEQUENCE,
}

px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = SEQUENCE
px.defaults.color_continuous_scale = [COLORS["neutral"], COLORS["accent"], COLORS["primary"]]


def style_fig(fig: go.Figure, title: str, width: int = 900, height: int = 480) -> go.Figure:
    """Applique les codes couleurs Wavestone à une figure Plotly."""
    fig.update_layout(
        title={"text": title, "x": 0.02, "xanchor": "left"},
        template={"layout": TEMPLATE_LAYOUT},
        width=width,
        height=height,
        margin={"l": 60, "r": 40, "t": 70, "b": 60},
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 1, "xanchor": "right"},
    )
    fig.update_xaxes(showgrid=True, gridcolor="#ECECEC", zeroline=False)
    fig.update_yaxes(showgrid=True, gridcolor="#ECECEC", zeroline=False)
    return fig

# Template EDA

Template générique d'analyse exploratoire et de modélisation pour un problème de **classification supervisée**.
Adapter les paramètres de la première cellule (`DATA_PATH`, `TARGET_COL`, `ID_COLS`, `POSITIVE_CLASS`),
puis exécuter le notebook de haut en bas et compléter les cellules de synthèse marquées « à compléter ».

Plan :

1. **Dataset** — stats générales, identification des features qualitatives / quantitatives (`col_qual`, `col_quant`),
   valeurs manquantes et **liens entre manquants**, analyse de la variable cible (barplot)
2. **Analyse univariée** — barplots en nombre et en % (qualitatives), boxplots et distributions (quantitatives)
3. **Analyse bivariée** — taux de classe positive par modalité (qualitatives), distributions par bin
   et bins métier (quantitatives)
4. **Modèle naïf V0** — XGBoost, CatBoost, régression logistique (validation croisée 5 blocs)
5. **Feature engineering** — guidé par les constats de l'EDA et des valeurs SHAP
6. **Modèle V1** — même protocole, features enrichies, comparaison V0 / V1
7. **Valeurs SHAP** — lecture des contributions par variable (beeswarm + importances)

## 1. Dataset

### 1.1 Chargement et aperçu

In [ ]:
data = pd.read_csv(DATA_PATH)


def coerce_numeric(df: pd.DataFrame, seuil: float = 0.9) -> pd.DataFrame:
    """Convertit en numérique les colonnes texte dont la quasi-totalité des valeurs sont
    numériques (gère les 'NA', ' ' et autres libellés importés en objet)."""
    for col in df.select_dtypes(include="object"):
        converted = pd.to_numeric(df[col], errors="coerce")
        if converted.notna().mean() >= seuil:
            df[col] = converted
    return df


data = coerce_numeric(data)

print(f"{data.shape[0]} lignes x {data.shape[1]} variables")
print(f"Lignes dupliquées : {data.duplicated().sum()}")
data.head()

In [ ]:
def overview(df: pd.DataFrame) -> pd.DataFrame:
    """Tableau de synthèse par variable : type, complétude, cardinalité, exemples."""
    rows = []
    for col in df.columns:
        rows.append({
            "variable": col,
            "dtype": str(df[col].dtype),
            "n_manquants": int(df[col].isna().sum()),
            "pct_manquants": round(100 * df[col].isna().mean(), 2),
            "n_modalites": int(df[col].nunique()),
            "exemples": ", ".join(map(str, df[col].dropna().unique()[:3])),
        })
    return pd.DataFrame(rows)


# Stats descriptives des variables numériques
display(data.describe().T.round(2))

# Synthèse par variable
overview(data)

### 1.2 Identification des features qualitatives et quantitatives

Objectif : constituer `col_quant` (variables quantitatives) et `col_qual` (variables qualitatives),
utilisées dans toute la suite de l'analyse.

In [ ]:
def split_quant_qual(df: pd.DataFrame, exclude: list, max_modalites: int = 10) -> tuple:
    """Sépare les colonnes en quantitatives / qualitatives.

    Heuristique : numérique et plus de `max_modalites` valeurs distinctes -> quantitative.
    À ajuster au contexte : une note de satisfaction de 0 à 10 est numérique mais se
    traite souvent comme qualitative (baisser `max_modalites` pour la basculer).
    """
    cols = [c for c in df.columns if c not in exclude]
    col_quant = [
        c for c in cols
        if pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique(dropna=True) > max_modalites
    ]
    col_qual = [c for c in cols if c not in col_quant]
    return col_quant, col_qual


col_quant, col_qual = split_quant_qual(data, exclude=ID_COLS + [TARGET_COL])
print(f"col_quant ({len(col_quant)}) : {col_quant}")
print(f"col_qual  ({len(col_qual)}) : {col_qual}")

### 1.3 Valeurs manquantes et liens entre manquants

Deux lectures complémentaires :

- **volume** : taux de manquants par variable ;
- **structure** : les manquants sont-ils liés entre eux ? Si `A` est manquante, `B` l'est-elle aussi
  sur les mêmes lignes ?

  Exemple assurance : si `hasLifeInsurance` est vide, alors `lifeInsuranceMonthlyCharge` est
  probablement vide aussi ; en revanche si `hasCarInsurance` est renseignée,
  `carInsuranceMonthlyCharge` ne devrait pas être vide.

  La table `missing_links` répond exactement à ces deux questions avec
  `P(B manquant | A manquant)` et `P(B manquant | A renseigné)`.

In [ ]:
def plot_missing(df: pd.DataFrame) -> go.Figure:
    """Taux de valeurs manquantes par variable."""
    miss = df.isna().mean().mul(100)
    miss = miss[miss > 0].sort_values(ascending=True)
    if miss.empty:
        print("Aucune valeur manquante.")
        return go.Figure()
    fig = px.bar(x=miss.values, y=miss.index, orientation="h",
                 labels={"x": "% de valeurs manquantes", "y": ""})
    fig.update_traces(marker_color=COLORS["primary"], showlegend=False)
    return style_fig(fig, "Valeurs manquantes par variable (%)", width=850, height=420)


def plot_null_correlation(df: pd.DataFrame) -> go.Figure:
    """Corrélation entre indicateurs de manquants : proche de 1 = manquants conjoints,
    proche de -1 = manquants exclusifs, proche de 0 = manquants indépendants."""
    cols_nan = [c for c in df.columns if df[c].isna().any()]
    if not cols_nan:
        return go.Figure()
    corr = df[cols_nan].isna().corr()
    fig = go.Figure(go.Heatmap(
        z=corr.values, x=corr.columns, y=corr.columns,
        colorscale=[[0, COLORS["negative"]], [0.5, COLORS["bg_light"]], [1, COLORS["primary"]]],
        zmin=-1, zmax=1, text=corr.round(2).values, texttemplate="%{text}",
        textfont={"size": 10},
    ))
    fig.update_xaxes(tickangle=-45)
    return style_fig(fig, "Corrélation des manquants (variables avec NaN)", width=850, height=650)


plot_missing(data).show()
plot_null_correlation(data).show()

In [ ]:
def missing_links(df: pd.DataFrame, seuil: float = 0.5) -> pd.DataFrame:
    """Pour chaque variable A contenant des manquants, liste les variables B dont les
    manquants sont alignés sur les lignes de A (P(B manquant | A manquant) >= seuil).

    Lecture attendue :
    - P(B manquant | A manquant) élevée et P(B manquant | A renseigné) proche de 0 :
      structure de manquants **liés** (ex. montant d'une option non souscrite vide quand
      l'option est absente) -> à traiter lors du feature engineering par une modalité
      explicite ou un indicateur, pas par une imputation aveugle ;
    - une P(B manquant | A renseigné) élevée signale un problème de qualité de données
      (valeur qui devrait être renseignée mais ne l'est pas).
    """
    cols_nan = [c for c in df.columns if df[c].isna().any()]
    rows = []
    for a in cols_nan:
        mask_a = df[a].isna()
        for b in df.columns:
            if b == a:
                continue
            p_lien = df.loc[mask_a, b].isna().mean()
            if p_lien >= seuil:
                rows.append({
                    "A (manquante)": a,
                    "n_manquants_A": int(mask_a.sum()),
                    "B (liée)": b,
                    "P(B manquant | A manquant)": round(p_lien, 2),
                    "P(B manquant | A renseigné)": round(df.loc[~mask_a, b].isna().mean(), 2),
                })
    if not rows:
        print(f"Aucune liaison entre manquants au seuil de {seuil:.0%}.")
        return pd.DataFrame()
    return (
        pd.DataFrame(rows)
        .sort_values("P(B manquant | A manquant)", ascending=False)
        .reset_index(drop=True)
    )


missing_links(data, seuil=0.5)

### 1.4 Variable cible

Vérifications attendues sur la cible : présence de manquants (lignes à exclure) et
**déséquilibre des classes** — un déséquilibre impose une métrique adaptée (AUC PR,
baseline = prévalence de la classe positive) et une validation croisée stratifiée.

In [ ]:
# Lignes sans cible : exclues (ni entraînables ni évaluables)
n_target_nan = data[TARGET_COL].isna().sum()
if n_target_nan:
    print(f"{n_target_nan} lignes sans cible -> exclues")
    data = data.loc[data[TARGET_COL].notna()].reset_index(drop=True)


def plot_target_classification(df: pd.DataFrame, target: str, positive_class) -> go.Figure:
    """Barplot de répartition de la cible (effectifs et %)."""
    counts = df[target].value_counts(dropna=False)
    pct = df[target].value_counts(normalize=True, dropna=False).mul(100).round(1)
    fig = go.Figure()
    for label in counts.index:
        fig.add_trace(go.Bar(
            x=[str(label)], y=[counts[label]],
            text=[f"{counts[label]} ({pct[label]} %)"], textposition="outside",
            marker_color=COLORS["negative"] if label == positive_class else COLORS["positive"],
            showlegend=False,
        ))
    fig.update_layout(yaxis_title="Nombre de lignes")
    return style_fig(fig, f"Répartition de la variable cible : {target}", width=700, height=450)


POSITIVE_RATE = (data[TARGET_COL] == POSITIVE_CLASS).mean()
print(f"Taux de classe positive '{POSITIVE_CLASS}' : {100 * POSITIVE_RATE:.1f} %")
plot_target_classification(data, TARGET_COL, POSITIVE_CLASS).show()

### Synthèse dataset (à compléter)

Questions auxquelles répondre ici :

- volume, doublons, qualité générale des données ;
- répartition `col_quant` / `col_qual` retenue (et ajustements de l'heuristique) ;
- structure des manquants : colonnes liées, stratégie à prévoir pour le feature engineering ;
- cible : déséquilibre des classes et implication sur le choix de la métrique (AUC PR, baseline = prévalence) ;
- cible : asymétrie / transformation éventuelle (log), outliers ;

*(Conserver uniquement la ligne utile selon le cas.)*

## 2. Analyse univariée

### 2.1 Variables qualitatives — barplots en nombre et en %

In [ ]:
def clean_labels(index) -> list:
    """Modalités lisibles (manquants explicites)."""
    return ["manquant" if pd.isna(m) else str(m) for m in index]


def plot_qual_grid(df: pd.DataFrame, cols: list, mode: str = "nb", ncols: int = 4) -> go.Figure:
    """Grille de barres horizontales par modalité : effectifs (mode='nb') ou fréquences (mode='pct')."""
    if not cols:
        print("Aucune variable qualitative.")
        return go.Figure()
    ncols = min(ncols, len(cols))
    nrows = -(-len(cols) // ncols)
    fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=cols)
    for i, col in enumerate(cols):
        counts = df[col].value_counts(dropna=False)
        values = counts.values if mode == "nb" else (100 * counts / counts.sum()).round(1)
        fig.add_trace(
            go.Bar(x=values, y=clean_labels(counts.index), orientation="h",
                   marker_color=SEQUENCE[i % len(SEQUENCE)], showlegend=False),
            row=i // ncols + 1, col=i % ncols + 1,
        )
    fig.update_annotations(font={"size": 13})
    suffix = "effectifs" if mode == "nb" else "fréquences (%)"
    return style_fig(fig, f"Variables qualitatives — {suffix}", width=1250, height=280 * nrows)


plot_qual_grid(data, col_qual, mode="nb").show()
plot_qual_grid(data, col_qual, mode="pct").show()

In [ ]:
# Tableau de synthèse : effectifs et % par modalité
if col_qual:
    qual_summary = (
        pd.melt(data, value_vars=col_qual)
        .groupby(["variable", "value"], observed=True, dropna=False)
        .size().rename("nb").reset_index()
    )
    qual_summary["pct"] = (
        100 * qual_summary["nb"] / qual_summary.groupby("variable")["nb"].transform("sum")
    ).round(1)
    display(qual_summary.sort_values(["variable", "nb"], ascending=[True, False]))
else:
    print("Aucune variable qualitative.")

### 2.2 Variables quantitatives — boxplots et distributions

In [ ]:
def plot_boxplot_grid(df: pd.DataFrame, cols: list, ncols: int = 3) -> go.Figure:
    """Grille de box plots (une variable quantitative par sous-graphe)."""
    if not cols:
        print("Aucune variable quantitative.")
        return go.Figure()
    ncols = min(ncols, len(cols))
    nrows = -(-len(cols) // ncols)
    fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=cols)
    for i, col in enumerate(cols):
        fig.add_trace(
            go.Box(x=df[col], marker_color=SEQUENCE[i % len(SEQUENCE)], showlegend=False),
            row=i // ncols + 1, col=i % ncols + 1,
        )
    fig.update_annotations(font={"size": 13})
    return style_fig(fig, "Box plots des variables quantitatives", width=1250, height=300 * nrows)


def plot_numeric(df: pd.DataFrame, col: str) -> go.Figure:
    """Distribution d'une variable quantitative : histogramme + box plot marginal."""
    fig = px.histogram(df, x=col, nbins=40, marginal="box")
    fig.update_traces(marker_color=COLORS["primary"], selector={"type": "histogram"})
    fig.update_layout(bargap=0.05, xaxis_title=col, yaxis_title="Nombre de lignes")
    return style_fig(fig, f"Distribution de {col}", width=850, height=420)


quant_summary = data[col_quant].describe().T
quant_summary["skew"] = data[col_quant].skew()
display(quant_summary.round(2))

plot_boxplot_grid(data, col_quant).show()
for col in col_quant:
    plot_numeric(data, col).show()

### Synthèse analyse univariée (à compléter)

Questions auxquelles répondre ici :

- modalités rares / variables constantes ou quasi constantes (candidates à l'écart) ;
- distributions multimodales (candidats à des bins métier pour l'analyse bivariée) ;
- asymétries fortes / valeurs extrêmes (transformations à prévoir pour les modèles linéaires).

## 3. Analyse bivariée — lien des features avec la cible (classification)

Objectif : quantifier le lien entre chaque feature et la **classe positive**.

- **features qualitatives** : taux de classe positive par modalité (barplot en %),
  hiérarchisé par test du Khi² et V de Cramér ;
- **features quantitatives** : distributions par classe, puis lecture par bin —
  effectifs et % de chaque classe au sein de chaque bin — et, pour les variables dont le
  lien est non monotone, **bins métier** pour lire le taux de classe positive par tranche.

### 3.1 Features qualitatives vs cible

In [ ]:
GLOBAL_RATE = 100 * (data[TARGET_COL] == POSITIVE_CLASS).mean()


def target_rate_by(df: pd.DataFrame, col: str) -> pd.DataFrame:
    """Effectifs et taux de classe positive (%) par modalité (manquants inclus)."""
    g = df.groupby(col, observed=True, dropna=False)[TARGET_COL]
    out = g.agg(nb="size", nb_pos=lambda s: (s == POSITIVE_CLASS).sum())
    out["taux_pos"] = 100 * out["nb_pos"] / out["nb"]
    out["pct_lignes"] = 100 * out["nb"] / len(df)
    return out


def plot_target_rate(df: pd.DataFrame, col: str, height: int = 420) -> go.Figure:
    """Taux de classe positive par modalité, trié, avec référence au taux global."""
    rates = target_rate_by(df, col).sort_values("taux_pos")
    fig = go.Figure(go.Bar(
        x=rates["taux_pos"], y=clean_labels(rates.index), orientation="h",
        text=[f"{r:.1f} % (n={n})" for r, n in zip(rates["taux_pos"], rates["nb"])],
        textposition="outside",
        marker_color=COLORS["primary"], showlegend=False,
    ))
    fig.add_vline(x=GLOBAL_RATE, line_dash="dash", line_color=COLORS["negative"],
                  annotation_text=f"taux global : {GLOBAL_RATE:.1f} %")
    fig.update_layout(xaxis_title=f"Taux de '{POSITIVE_CLASS}' (%)", yaxis_title="")
    return style_fig(fig, f"Taux de classe positive par {col}", width=850, height=height)


def plot_target_rate_grid(df: pd.DataFrame, cols: list, ncols: int = 4) -> go.Figure:
    """Grille : taux de classe positive par modalité pour chaque feature qualitative."""
    if not cols:
        return go.Figure()
    ncols = min(ncols, len(cols))
    nrows = -(-len(cols) // ncols)
    fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=cols)
    for i, col in enumerate(cols):
        rates = target_rate_by(df, col).sort_values("taux_pos")
        fig.add_trace(
            go.Bar(x=rates["taux_pos"], y=clean_labels(rates.index), orientation="h",
                   marker_color=COLORS["primary"], showlegend=False),
            row=i // ncols + 1, col=i % ncols + 1,
        )
        fig.add_vline(x=GLOBAL_RATE, line_dash="dot", line_color=COLORS["negative"],
                      row=i // ncols + 1, col=i % ncols + 1)
    fig.update_annotations(font={"size": 12})
    return style_fig(fig, "Taux de classe positive par modalité — features qualitatives",
                     width=1250, height=280 * nrows)


plot_target_rate_grid(data, col_qual).show()

In [ ]:
from scipy.stats import chi2_contingency


def chi2_cramers_v(df: pd.DataFrame, col: str) -> dict:
    """Test d'indépendance du Khi² et V de Cramér entre une feature qualitative et la
    cible (V de Cramér : 0 = indépendant, 1 = lien parfait)."""
    table = pd.crosstab(df[col].fillna("manquant"), df[TARGET_COL])
    chi2, p_value, dof, _ = chi2_contingency(table)
    n = table.to_numpy().sum()
    v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))
    return {"variable": col, "khi2": round(chi2, 1), "p_value": round(p_value, 4),
            "V_cramer": round(v, 3)}


assoc_qual = (
    pd.DataFrame([chi2_cramers_v(data, c) for c in col_qual])
    .sort_values("V_cramer", ascending=False)
    .reset_index(drop=True)
)


def plot_cramers_v(assoc_df: pd.DataFrame) -> go.Figure:
    """Classement des variables qualitatives par force du lien avec la cible."""
    fig = go.Figure(go.Bar(
        x=assoc_df["V_cramer"], y=assoc_df["variable"], orientation="h",
        text=assoc_df["V_cramer"], textposition="outside",
        marker_color=COLORS["primary"], showlegend=False,
    ))
    fig.update_layout(xaxis_title="V de Cramér (0 = indépendant, 1 = lien parfait)",
                      yaxis_title="")
    return style_fig(fig, "Force du lien avec la cible (V de Cramér)", width=850, height=520)


display(assoc_qual)
plot_cramers_v(assoc_qual[::-1]).show()

### 3.2 Features quantitatives vs cible

In [ ]:
CLASS_COLORS = {
    label: (COLORS["negative"] if label == POSITIVE_CLASS else COLORS["positive"])
    for label in data[TARGET_COL].unique()
}


def plot_numeric_by_target(df: pd.DataFrame, col: str, nbins: int = 40) -> go.Figure:
    """Distribution d'une variable quantitative selon la classe cible (superposition + box)."""
    fig = px.histogram(df, x=col, color=TARGET_COL, nbins=nbins, barmode="overlay",
                       opacity=0.7, marginal="box", color_discrete_map=CLASS_COLORS)
    fig.update_layout(bargap=0.05, xaxis_title=col, yaxis_title="Nombre de lignes")
    return style_fig(fig, f"{col} selon la classe cible", width=850, height=450)


for col in col_quant:
    plot_numeric_by_target(data, col).show()

# Premiers ordres de grandeur : moyenne / médiane de chaque variable par classe
display(data.groupby(TARGET_COL, observed=True)[col_quant].agg(["mean", "median"]).round(2))

In [ ]:
def plot_class_by_bin(df: pd.DataFrame, col: str, n_bins: int = 10,
                      strategy: str = "uniform") -> go.Figure:
    """Barplot par bin de `col` : effectifs de chaque classe et % de la classe au sein
    du bin (manquants exclus). strategy='uniform' -> bins de largeur égale,
    'quantile' -> bins d'effectifs comparables."""
    if strategy == "quantile":
        binned = pd.qcut(df[col], q=n_bins, duplicates="drop")
    else:
        binned = pd.cut(df[col], bins=n_bins)
    tab = (
        df.assign(_bin=binned)
        .groupby(["_bin", TARGET_COL], observed=True)
        .size().unstack(fill_value=0)
    )
    share = tab.div(tab.sum(axis=1), axis=0).mul(100).round(1)
    fig = go.Figure()
    for label in tab.columns:
        fig.add_trace(go.Bar(
            name=str(label), x=[str(b) for b in tab.index], y=tab[label],
            text=[f"{n} ({p:.0f} %)" for n, p in zip(tab[label], share[label])],
            textposition="outside",
            marker_color=CLASS_COLORS.get(label, COLORS["primary"]),
        ))
    fig.update_layout(barmode="group", xaxis_title=f"{col} (bins)",
                      yaxis_title="Nombre de lignes")
    return style_fig(fig, f"{col} — effectifs et % par bin et par classe", width=1000, height=480)


for col in col_quant:
    plot_class_by_bin(data, col).show()

In [ ]:
# Pour les features quantitatives dont la distribution est structurée (cf. 2.2) ou dont
# le lien avec la cible est non monotone (cf. 3.2) : définir des bins métier et lire le
# taux de classe positive par bin. À adapter.
CUSTOM_BINS = {
    # "tenure": {
    #     "edges": [-1, 12, 24, 48, 72],
    #     "labels": ["0-12 mois", "13-24 mois", "25-48 mois", "49-72 mois"],
    # },
}

for col, spec in CUSTOM_BINS.items():
    data[f"{col}_group"] = pd.cut(data[col], bins=spec["edges"], labels=spec["labels"])
    plot_target_rate(data, f"{col}_group").show()
    display(target_rate_by(data, f"{col}_group"))

if not CUSTOM_BINS:
    print("CUSTOM_BINS vide : renseigner les bins métier identifiés en analyse univariée.")

In [ ]:
# Corrélation (point-bisérale) des variables quantitatives avec la cible encodée 0/1
y_binary = (data[TARGET_COL] == POSITIVE_CLASS).astype(int)
corr_with_target = data[col_quant].corrwith(y_binary).sort_values()

fig = go.Figure(go.Bar(
    x=corr_with_target.values, y=corr_with_target.index, orientation="h",
    text=[f"{v:.2f}" for v in corr_with_target.values], textposition="outside",
    marker_color=[COLORS["negative"] if v > 0 else COLORS["positive"]
                  for v in corr_with_target.values],
    showlegend=False,
))
fig.update_layout(
    xaxis_title=f"Corrélation avec '{POSITIVE_CLASS}' (rouge = positif, vert = négatif)",
    yaxis_title="",
)
style_fig(fig, "Corrélation des variables quantitatives avec la cible",
         width=900, height=450).show()

### 3.3 Liens globaux entre features

Vue d'ensemble des **redondances entre features** et des **signaux avec la cible**, par type de variable :

- **features quantitatives** : matrice de corrélation de Pearson (colonnes quantitatives,
  cible incluse quand elle est quantitative), **top 10 des paires de features les plus
  corrélées** (en valeur absolue, multicolinéarité à surveiller) et **top 10 des
  corrélations en valeur absolue avec la cible** ;
- **features qualitatives** : matrice du **V de Cramér** (0 = indépendant, 1 = lien parfait,
  cible incluse quand elle est qualitative), **top 10 des paires de features les plus
  liées** et **top 10 des V de Cramér avec la cible**.

In [ ]:
def plot_correlation_matrix(corr: pd.DataFrame) -> go.Figure:
    """Heatmap de la matrice de corrélation (palette Wavestone)."""
    fig = go.Figure(go.Heatmap(
        z=corr.values, x=corr.columns, y=corr.columns,
        colorscale=[[0, COLORS["negative"]], [0.5, COLORS["bg_light"]], [1, COLORS["primary"]]],
        zmin=-1, zmax=1, text=corr.round(2).values, texttemplate="%{text}",
        textfont={"size": 10},
    ))
    fig.update_xaxes(tickangle=-45)
    return style_fig(fig, "Matrice de corrélation (Pearson) — variables quantitatives",
                     width=850, height=720)


def top_abs_pairs(matrix: pd.DataFrame, n: int = 10, exclude: list = (),
                  nom_valeur: str = "valeur") -> pd.DataFrame:
    """Top n des paires de variables les plus liées, en valeur absolue, dans une matrice
    symétrique (`exclude` permet d'écarter la cible des paires entre features)."""
    cols = [c for c in matrix.columns if c not in exclude]
    sub = matrix.loc[cols, cols]
    pairs = (
        sub.where(np.triu(np.ones(sub.shape, dtype=bool), k=1))
        .stack().rename(nom_valeur).reset_index()
    )
    pairs.columns = ["variable_1", "variable_2", nom_valeur]
    pairs["valeur_abs"] = pairs[nom_valeur].abs()
    return pairs.sort_values("valeur_abs", ascending=False).head(n).reset_index(drop=True)


def top_abs_with_target(values: pd.Series, n: int = 10,
                        nom_valeur: str = "valeur") -> pd.DataFrame:
    """Top n, en valeur absolue, du lien de chaque feature avec la cible."""
    out = pd.DataFrame({nom_valeur: values})
    out["valeur_abs"] = out[nom_valeur].abs()
    return out.sort_values("valeur_abs", ascending=False).head(n)


# --- Features quantitatives : matrice de corrélation (Pearson) ---------------
corr_matrix_quant = data[col_quant].corr(method="pearson")
plot_correlation_matrix(corr_matrix_quant).show()

print("Top 10 des paires de features quantitatives les plus corrélées (en valeur absolue)")
display(top_abs_pairs(corr_matrix_quant, n=10, nom_valeur="correlation").round(3))

# Cible qualitative -> corrélation point-bisérale avec la cible encodée 0/1
print(f"Top 10 des corrélations en valeur absolue avec '{TARGET_COL}' (point-bisérale)")
y_binary = (data[TARGET_COL] == POSITIVE_CLASS).astype(int)
display(top_abs_with_target(data[col_quant].corrwith(y_binary), n=10,
                            nom_valeur="correlation").round(3))

In [ ]:
def cramers_v(x: pd.Series, y: pd.Series) -> float:
    """V de Cramér entre deux variables qualitatives (0 = indépendant, 1 = lien parfait)."""
    table = pd.crosstab(x.fillna("manquant"), y.fillna("manquant"))
    if min(table.shape) < 2:
        return 0.0  # variable constante : association non testable
    chi2 = chi2_contingency(table)[0]
    n = table.to_numpy().sum()
    return float(np.sqrt(chi2 / (n * (min(table.shape) - 1))))


def cramers_v_matrix(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """Matrice du V de Cramér entre variables qualitatives (diagonale = 1)."""
    mat = pd.DataFrame(np.eye(len(cols)), index=cols, columns=cols)
    for i, a in enumerate(cols):
        for b in cols[i + 1:]:
            v = cramers_v(df[a], df[b])
            mat.loc[a, b] = v
            mat.loc[b, a] = v
    return mat


def plot_cramers_v_matrix(mat: pd.DataFrame) -> go.Figure:
    """Heatmap de la matrice du V de Cramér (palette Wavestone, échelle 0-1)."""
    fig = go.Figure(go.Heatmap(
        z=mat.values, x=mat.columns, y=mat.columns,
        colorscale=[[0, COLORS["bg_light"]], [1, COLORS["primary"]]],
        zmin=0, zmax=1, text=mat.round(2).values, texttemplate="%{text}",
        textfont={"size": 10},
    ))
    fig.update_xaxes(tickangle=-45)
    return style_fig(fig, "Matrice du V de Cramér — features qualitatives",
                     width=850, height=720)


# --- Features qualitatives : matrice du V de Cramér (cible qualitative incluse)
if col_qual:
    cramer_matrix = cramers_v_matrix(data, col_qual + [TARGET_COL])
    plot_cramers_v_matrix(cramer_matrix).show()

    print("Top 10 des paires de features qualitatives les plus liées (V de Cramér)")
    display(top_abs_pairs(cramer_matrix, n=10, exclude=[TARGET_COL],
                          nom_valeur="V_cramer").round(3))

    print(f"Top 10 des V de Cramér avec '{TARGET_COL}'")
    display(top_abs_with_target(cramer_matrix[TARGET_COL].drop(TARGET_COL), n=10,
                                nom_valeur="V_cramer").round(3))
else:
    print("Aucune variable qualitative.")

### Synthèse analyse bivariée (à compléter)

Questions auxquelles répondre ici :

- quelles features sont significativement liées à la cible (p-value < 0.05) et avec quelle
  force (V de Cramér / corrélation) ?
- quelles modalités / tranches se détachent du taux global ?
- quelles relations sont non monotones (bins métier à créer) ?
- quelles redondances entre features (multicolinéarité) pour la modélisation ?

## 4. Premier modèle naïf (V0)

Trois familles de modèles, **sans optimisation d'hyperparamètres ni feature engineering**,
pour établir une référence :

- **XGBoost** et **CatBoost** : gradient boosting, gèrent nativement les valeurs manquantes ;
- **régression logistique** : référence linéaire et interprétable — imputation et standardisation
  intégrées dans un pipeline scikit-learn, appliquées pli par pli (aucune fuite de données).

Protocole : one-hot sur les features qualitatives, validation croisée **stratifiée** à 5 plis
(chaque pli conserve la proportion de classes). Métriques : **AUC PR** (principale, baseline =
prévalence de la classe positive) et **AUC ROC** (secondaire, baseline = 0.5).

Les colonnes `_group` créées en 3.2 restent dans `data` : les écarter via `DROP_COLS`
si elles dupliquent leur variable d'origine.

In [ ]:
from catboost import CatBoostClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

# Colonnes écartées du modèle : identifiants + cible (+ colonnes dérivées redondantes)
DROP_COLS = ID_COLS + [TARGET_COL]


def prepare_X(df: pd.DataFrame, drop_cols: list) -> pd.DataFrame:
    """Matrice de features : one-hot sur les qualitatives, quantitatives telles quelles."""
    X = df.drop(columns=[c for c in drop_cols if c in df.columns])
    return pd.get_dummies(X, dtype=float)


MODELS_V0 = {
    "XGBoost": XGBClassifier(random_state=42, eval_metric="logloss"),
    "CatBoost": CatBoostClassifier(random_state=42, verbose=0),
    "Rég. logistique": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]),
}


def evaluate_classifiers(models: dict, X: pd.DataFrame, y: pd.Series,
                        n_splits: int = 5) -> pd.DataFrame:
    """Validation croisée stratifiée : AUC PR et AUC ROC par modèle et par pli."""
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    records = []
    for name, model in models.items():
        res = cross_validate(model, X, y, cv=cv,
                             scoring={"auc_pr": "average_precision",
                                      "auc_roc": "roc_auc"})
        records.append({
            "modele": name,
            "AUC_PR_plis": res["test_auc_pr"],
            "AUC_ROC_plis": res["test_auc_roc"],
        })
    return pd.DataFrame(records)


def summarize_cv(cv_records: pd.DataFrame) -> pd.DataFrame:
    """Moyenne et écart-type des métriques par modèle."""
    rows = []
    for _, rec in cv_records.iterrows():
        rows.append({
            "modele": rec["modele"],
            "AUC_PR_moyenne": rec["AUC_PR_plis"].mean(),
            "AUC_PR_ecart_type": rec["AUC_PR_plis"].std(),
            "AUC_ROC_moyenne": rec["AUC_ROC_plis"].mean(),
            "AUC_ROC_ecart_type": rec["AUC_ROC_plis"].std(),
        })
    return pd.DataFrame(rows).set_index("modele").round(4)


X = prepare_X(data, DROP_COLS)
y = (data[TARGET_COL] == POSITIVE_CLASS).astype(int)
print(f"{X.shape[0]} lignes x {X.shape[1]} variables après one-hot")

cv_v0 = evaluate_classifiers(MODELS_V0, X, y)
summary_v0 = summarize_cv(cv_v0)
summary_v0

In [ ]:
def plot_model_comparison(summary: pd.DataFrame, metrics: list, title: str,
                         baseline: float | None = None) -> go.Figure:
    """Barplot groupé : performance des modèles pour chaque métrique."""
    fig = go.Figure()
    for name, color in zip(summary.index,
                           [COLORS["accent"], COLORS["neutral_dark"], COLORS["primary"]]):
        values = [summary.loc[name, m] for m in metrics]
        fig.add_trace(go.Bar(
            name=name, x=metrics, y=values,
            text=[f"{v:.3f}" for v in values], textposition="outside",
            marker_color=color,
        ))
    if baseline is not None:
        fig.add_hline(y=baseline, line_dash="dash", line_color=COLORS["negative"],
                      annotation_text=f"baseline AUC PR (prévalence) : {baseline:.3f}")
    fig.update_layout(barmode="group", yaxis_title="Score", yaxis_range=[0, 1])
    return style_fig(fig, title, width=850, height=420)


plot_model_comparison(
    summary_v0, ["AUC_PR_moyenne", "AUC_ROC_moyenne"],
    title="V0 — comparaison des 3 modèles (moyenne sur 5 plis)",
    baseline=POSITIVE_RATE,
).show()

### Synthèse V0 (à compléter)

Questions auxquelles répondre ici :

- quel modèle familial sert de référence ? (ordre de grandeur de la métrique principale
  face à sa baseline)
- les plis sont-ils homogènes (faible écart-type) ou y a-t-il une instabilité à creuser ?
- cohérence avec l'analyse bivariée : le modèle a-t-il appris les relations attendues ?

## 5. Feature engineering

Checklist guidée par les constats de l'EDA :

1. **Manquants liés** (cf. 1.3 `missing_links`) : expliciter par une modalité (`fillna`)
   ou un indicateur (`isna`) plutôt qu'imputer aveuglément ;
2. **Bins métier** (cf. 3) : transformer les relations non linéaires en features
   catégorielles lisibles ;
3. **Interactions** : croiser les variables dont les effets sont conjoints
   (cf. corrélations et valeurs SHAP de la V0) ;
4. **Écarts** : variables constantes / quasi constantes, variables redondantes
   (corrélation élevée), fuites de données potentielles ;
5. **Modèles linéaires** : imputation + échelle restent dans le pipeline, rien à gérer ici.

In [ ]:
data_v1 = data.copy()

# --- 1) Manquants liés (cf. missing_links) ----------------------------------
# Exemple : les manquants de X correspondent à une absence de souscription
# data_v1["X"] = data_v1["X"].fillna("Non souscrit")

# --- 2) Bins métier (cf. analyse bivariée) ----------------------------------
# data_v1["col_group"] = pd.cut(data_v1["col"], bins=[...], labels=[...])

# --- 3) Interactions --------------------------------------------------------
# data_v1["colA_x_colB"] = data_v1["colA"] * data_v1["colB"]

# --- 4) Variables à écarter -------------------------------------------------
FE_DROP_COLS: list = []  # ex. ["variable_constante", "variable_redondante"]

print(f"{data_v1.shape[0]} lignes x {data_v1.shape[1]} variables après feature engineering")
data_v1.head()

## 6. Modèle V1

Même protocole que la V0 — mais features engineering & potentielle optimisation d'hyperparamètre

In [ ]:
X_v1 = prepare_X(data_v1, DROP_COLS + FE_DROP_COLS)
y = (data_v1[TARGET_COL] == POSITIVE_CLASS).astype(int)
print(f"{X.shape[1]} variables (V0) -> {X_v1.shape[1]} variables (V1)")

cv_v1 = evaluate_classifiers(MODELS_V0, X_v1, y)
summary_v1 = summarize_cv(cv_v1)
summary_v1

In [ ]:
def plot_v0_vs_v1(summary_v0: pd.DataFrame, summary_v1: pd.DataFrame,
                  metric: str, title: str) -> go.Figure:
    """Barplot groupé : métrique principale des modèles en V0 et V1."""
    fig = go.Figure()
    for stage, summary, color in [("V0", summary_v0, COLORS["accent"]),
                                  ("V1", summary_v1, COLORS["primary"])]:
        fig.add_trace(go.Bar(
            name=stage, x=summary.index, y=summary[metric],
            text=[f"{v:.3f}" for v in summary[metric]], textposition="outside",
            marker_color=color,
        ))
    if metric == "AUC_PR_moyenne":
        fig.add_hline(y=POSITIVE_RATE, line_dash="dash", line_color=COLORS["negative"],
                      annotation_text=f"baseline AUC PR (prévalence) : {POSITIVE_RATE:.3f}")
    fig.update_layout(barmode="group", yaxis_title=metric, yaxis_range=[0, 1])
    return style_fig(fig, title, width=850, height=420)


plot_v0_vs_v1(summary_v0, summary_v1, "AUC_PR_moyenne",
              "V0 vs V1 — AUC PR (moyenne sur 5 plis)").show()

### Synthèse V1 (à compléter)

Questions auxquelles répondre ici :

- quel gain apporte le feature engineering sur la métrique principale ?
- les gains sont-ils réguliers sur les 5 plis ?
- quel modèle retenir pour l'analyse SHAP et l'optimisation ?

## 7. Analyse des valeurs SHAP

SHAP décompose chaque prédiction en contributions par variable : une valeur positive pousse
vers la classe positive, négative vers l'autre classe. Deux graphiques :

1. **beeswarm** : un point par ligne ; position horizontale = contribution, couleur = valeur
   de la variable (rouge = élevée, bleu = faible) ;
2. **importance globale** : moyenne des |SHAP| par variable.

Modèle à expliquer : celui retenu à l'issue de la V1 (`BEST_MODEL`). Les boosters sont
expliqués par `TreeExplainer`, les modèles linéaires par `LinearExplainer` (valeurs exactes) ;
pour un pipeline, l'explication se fait dans l'espace transformé (imputation + échelle).

In [ ]:
import shap

BEST_MODEL = "XGBoost"  # <- à adapter : modèle retenu à l'issue de la V1

model = MODELS_V0[BEST_MODEL]
model.fit(X_v1, y)

# Un pipeline impose d'expliquer dans l'espace transformé (imputer / scaler appliqués)
if isinstance(model, Pipeline):
    preprocessor = Pipeline(model.steps[:-1])
    X_explain = pd.DataFrame(preprocessor.transform(X_v1), columns=X_v1.columns,
                             index=X_v1.index)
    estimator = model.steps[-1][1]
else:
    X_explain = X_v1
    estimator = model

explainer = shap.Explainer(estimator, X_explain)
shap_values = explainer(X_explain)
print(f"Valeurs SHAP : {shap_values.shape[0]} lignes x {shap_values.shape[1]} variables")

shap.plots.beeswarm(shap_values, max_display=15)

In [ ]:
# Importance globale : moyenne des |SHAP| par variable
mean_abs_shap = (
    pd.Series(np.abs(shap_values.values).mean(axis=0), index=X_explain.columns,
              name="mean_abs_shap")
    .sort_values(ascending=False)
)
display(mean_abs_shap.head(15).round(3))

shap.plots.bar(shap_values, max_display=15)

### Lecture des valeurs SHAP (à compléter)

Questions auxquelles répondre ici :

- quelles variables dominent l'explication du modèle ?
- quelles valeurs extrêmes poussent vers la classe positive / négative ?
- cohérence avec l'analyse bivariée (taux par modalité, corrélations) ?
- pistes d'itération : interactions, bins, variables à écarter (retour à l'étape 5).

# Prochaines étapes

- optimisation des hyperparamètres du modèle retenu (grille / recherche aléatoire) ;
- choix du **seuil de décision** selon la capacité d'action des équipes métier
  (précision / rappel / f1-score) ;
- itérations de feature engineering guidées par les valeurs SHAP ;
- questions client ouvertes : unité des variables, signification des manquants, capacité
  d'action sur les lignes prédites positives. Analyse par (types de features, classe 1), exl analyse des features démographiques par rapport à la classe 1.